# Notebook 04.5: Pix2Pix Conditional GAN for Image Colorization

### 🎨 Why Pix2Pix Conditional GANs Excel at Colorization
Standard regression models (trained with $L_1$ or MSE loss alone) often suffer from the **regression-to-the-mean** phenomenon:
- For ambiguous surfaces (e.g., clothing, cars, houses), the ground truth color could be red, blue, or green.
- A standard $L_1$ loss averages these probabilities, resulting in **sepia, muted, or washed-out brownish tones**.

### ⚡ The Pix2Pix Architecture (Isola et al.):
1. **Conditional Generator ($G$):**
   - Takes the grayscale **$L$-channel ($1 \times 256 \times 256$)** as input.
   - Predicts the **$ab$-color channels ($2 \times 256 \times 256$)** bounded to $[-1, 1]$ via `Tanh`.
   - Uses a **U-Net** with skip connections (leveraging a pretrained ResNet-18 encoder) so fine spatial textures from $L$ are passed directly to the output.

2. **Conditional 70x70 PatchGAN Discriminator ($D$):**
   - Takes the concatenated pair: Grayscale $L$ + Color $ab$ = **$3$ input channels**.
   - Instead of classifying the entire image as real/fake, it classifies **$70 \times 70$ local patches** into real or synthetic.
   - This forces the Generator to produce high-frequency, authentic color textures rather than blurry averages.

3. **Composite Objective Function:**
   $$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{cGAN}}(G, D) + \lambda_{L_1} \mathcal{L}_{L_1}(G) + \lambda_{\text{VGG}} \mathcal{L}_{\text{VGG}}(G)$$
   - $\mathcal{L}_{\text{cGAN}}$ penalizes unrealistic color patterns.
   - $\mathcal{L}_{L_1}$ ensures structural alignment with the ground truth (weight $\lambda_{L_1} = 100.0$).
   - $\mathcal{L}_{\text{VGG}}$ aligns semantic feature representations.

### 1. Imports, Setup & Hardware Detection

In [ ]:
import os
import sys
# Add project root to sys.path
sys.path.append(os.path.abspath(".."))

import time
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm

# Custom project modules
from src.dataset import make_dataloaders, lab_to_rgb
from src.model import ResNetColorizationNet
from src.pix2pix import PatchGANDiscriminator, GANLoss
from src.loss import ColorizationLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

### 2. Dataset & DataLoaders Configuration

In [ ]:
train_dir = os.path.join("..", "data", "raw", "train")
val_dir = os.path.join("..", "data", "raw", "val")

train_paths = [os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith((".jpg", ".png"))]
val_paths = [os.path.join(val_dir, f) for f in os.listdir(val_dir) if f.endswith((".jpg", ".png"))]

BATCH_SIZE = 16
train_loader, val_loader = make_dataloaders(train_paths, val_paths, batch_size=BATCH_SIZE, num_workers=0)

print(f"Loaded {len(train_paths)} training images ({len(train_loader)} batches)")
print(f"Loaded {len(val_paths)} validation images ({len(val_loader)} batches)")

### 3. Initialize Pix2Pix Generator and PatchGAN Discriminator
- **Generator ($G$):** ResNet18-backbone U-Net (Pretrained weights speed up convergence significantly).
- **Discriminator ($D$):** 70x70 PatchGAN (Conditioned on $L + ab = 3$ channels).

In [ ]:
# 1. Initialize Generator and Discriminator
net_G = ResNetColorizationNet(pretrained=True).to(device)
net_D = PatchGANDiscriminator(in_channels=3, num_filters=64).to(device)

# Print model capacities
g_params = sum(p.numel() for p in net_G.parameters() if p.requires_grad)
d_params = sum(p.numel() for p in net_D.parameters() if p.requires_grad)
print(f"Generator Parameters:     {g_params:,}")
print(f"Discriminator Parameters: {d_params:,}")

# Quick sanity check forward pass
dummy_L = torch.randn(2, 1, 256, 256, device=device)
dummy_ab = torch.randn(2, 2, 256, 256, device=device)
fake_ab = net_G(dummy_L)
d_out = net_D(dummy_L, fake_ab)
print(f"Generated ab shape:       {fake_ab.shape}")
print(f"Discriminator patch shape:{d_out.shape} (30x30 receptive patch grid)")

### 4. Loss Functions, Optimizers & Schedulers
- **Adversarial Loss:** BCEWithLogitsLoss with one-sided label smoothing ($0.9$ for real).
- **L1 Loss:** Direct pixel-to-pixel color distance ($\lambda_{L_1} = 100.0$).
- **VGG Perceptual Loss:** High-level feature correspondence ($\lambda_{\text{VGG}} = 0.1$).
- **Optimizers:** Adam with $\beta_1 = 0.5, \beta_2 = 0.999$, learning rate $= 2 \times 10^{-4}$ (standard DCGAN/Pix2Pix setting).

In [ ]:
# Hyperparameters
LR_G = 2e-4
LR_D = 2e-4
BETA1 = 0.5
BETA2 = 0.999
LAMBDA_L1 = 100.0
USE_VGG = True

# Losses
criterion_GAN = GANLoss(gan_mode="vanilla", real_label=0.9, fake_label=0.0).to(device)
criterion_L1 = nn.L1Loss().to(device)
criterion_VGG = ColorizationLoss(vgg_weight=0.10).to(device) if USE_VGG else None

# Optimizers
optimizer_G = optim.Adam(net_G.parameters(), lr=LR_G, betas=(BETA1, BETA2))
optimizer_D = optim.Adam(net_D.parameters(), lr=LR_D, betas=(BETA1, BETA2))

# Mixed Precision Scalers (Tensor Core acceleration)
scaler_G = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
scaler_D = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

print("Configured Pix2Pix Loss & Optimizers successfully.")

### 5. Pix2Pix Training Loop

> **IMPORTANT NOTE:**  
> As requested, this cell is **not executed automatically** so you can run it whenever you are ready!  
> Each epoch takes approx. 30–45 seconds with AMP on the RTX 4050 GPU (~15 minutes for 20 epochs).

#### In each step:
1. **Discriminator Update:**
   - Real loss: $D(L, \text{real\_ab}) \to \text{Target: } 0.9$
   - Fake loss: $D(L, \text{fake\_ab.detach()}) \to \text{Target: } 0.0$
   - $\mathcal{L}_D = (\mathcal{L}_{D\text{-real}} + \mathcal{L}_{D\text{-fake}}) \times 0.5$
2. **Generator Update:**
   - Adversarial loss: $D(L, \text{fake\_ab}) \to \text{Target: } 1.0$
   - Color loss: $\|\text{fake\_ab} - \text{real\_ab}\|_1 \times 100.0$
   - Optional VGG perceptual loss
   - $\mathcal{L}_G = \mathcal{L}_{\text{adv}} + \lambda_{L1} \mathcal{L}_{L1} + \lambda_{\text{VGG}} \mathcal{L}_{\text{VGG}}$

In [ ]:
NUM_EPOCHS = 20
CHECKPOINT_DIR = os.path.join("..", "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "best_pix2pix_generator.pth")

def train_one_epoch_pix2pix(epoch, net_G, net_D, loader, opt_G, opt_D, scaler_G, scaler_D):
    net_G.train()
    net_D.train()
    
    running_loss_D = 0.0
    running_loss_G = 0.0
    running_loss_L1 = 0.0
    
    loop = tqdm(loader, desc=f"Epoch [{epoch+1}/{NUM_EPOCHS}]", leave=True)
    for batch in loop:
        real_L = batch["L"].to(device)
        real_ab = batch["ab"].to(device)
        
        # -----------------------------
        # (1) Train Discriminator D
        # -----------------------------
        opt_D.zero_grad()
        
        if scaler_D:
            with torch.amp.autocast('cuda'):
                # Real pair: (L, real_ab)
                pred_real = net_D(real_L, real_ab)
                loss_D_real = criterion_GAN(pred_real, is_real=True)
                
                # Fake pair: (L, fake_ab.detach())
                fake_ab = net_G(real_L)
                pred_fake = net_D(real_L, fake_ab.detach())
                loss_D_fake = criterion_GAN(pred_fake, is_real=False)
                
                loss_D = (loss_D_real + loss_D_fake) * 0.5
            
            scaler_D.scale(loss_D).backward()
            scaler_D.step(opt_D)
            scaler_D.update()
        else:
            pred_real = net_D(real_L, real_ab)
            loss_D_real = criterion_GAN(pred_real, is_real=True)
            fake_ab = net_G(real_L)
            pred_fake = net_D(real_L, fake_ab.detach())
            loss_D_fake = criterion_GAN(pred_fake, is_real=False)
            loss_D = (loss_D_real + loss_D_fake) * 0.5
            loss_D.backward()
            opt_D.step()
            
        # -----------------------------
        # (2) Train Generator G
        # -----------------------------
        opt_G.zero_grad()
        
        if scaler_G:
            with torch.amp.autocast('cuda'):
                # G wants D to classify fake_ab as real (1.0)
                pred_fake_for_G = net_D(real_L, fake_ab)
                loss_G_GAN = criterion_GAN(pred_fake_for_G, is_real=True)
                
                # L1 distance
                loss_G_L1 = criterion_L1(fake_ab, real_ab) * LAMBDA_L1
                
                # Optional VGG loss
                if criterion_VGG:
                    _, _, loss_vgg = criterion_VGG(real_L, fake_ab, real_ab)
                    loss_G = loss_G_GAN + loss_G_L1 + loss_vgg
                else:
                    loss_G = loss_G_GAN + loss_G_L1
            
            scaler_G.scale(loss_G).backward()
            scaler_G.step(opt_G)
            scaler_G.update()
        else:
            pred_fake_for_G = net_D(real_L, fake_ab)
            loss_G_GAN = criterion_GAN(pred_fake_for_G, is_real=True)
            loss_G_L1 = criterion_L1(fake_ab, real_ab) * LAMBDA_L1
            loss_G = loss_G_GAN + loss_G_L1
            loss_G.backward()
            opt_G.step()
            
        running_loss_D += loss_D.item()
        running_loss_G += loss_G.item()
        running_loss_L1 += loss_G_L1.item() / LAMBDA_L1
        
        loop.set_postfix({
            "D_loss": f"{loss_D.item():.4f}",
            "G_loss": f"{loss_G.item():.4f}",
            "L1": f"{(loss_G_L1.item() / LAMBDA_L1):.4f}"
        })
        
    num_batches = len(loader)
    return running_loss_D / num_batches, running_loss_G / num_batches, running_loss_L1 / num_batches

print("Training function compiled and ready. Run below loop cell when ready to train.")

### 6. Execute Training Loop (Run When Ready)
- Uncomment and execute the cell below when you are ready to begin training.

In [ ]:
# # To train, simply uncomment the following lines and run this cell:
# 
# best_val_l1 = float("inf")
# for epoch in range(NUM_EPOCHS):
#     d_loss, g_loss, l1_loss = train_one_epoch_pix2pix(
#         epoch, net_G, net_D, train_loader, optimizer_G, optimizer_D, scaler_G, scaler_D
#     )
#     print(f"Epoch {epoch+1:02d} Summary | Loss D: {d_loss:.4f} | Loss G: {g_loss:.4f} | Mean L1: {l1_loss:.4f}")
#     
#     # Save checkpoint if improved
#     if l1_loss < best_val_l1:
#         best_val_l1 = l1_loss
#         torch.save({
#             "epoch": epoch + 1,
#             "generator_state_dict": net_G.state_dict(),
#             "discriminator_state_dict": net_D.state_dict(),
#             "optimizer_G_state_dict": optimizer_G.state_dict(),
#             "optimizer_D_state_dict": optimizer_D.state_dict(),
#             "l1_loss": l1_loss
#         }, BEST_CKPT_PATH)
#         print(f"  --> Saved new best checkpoint to {BEST_CKPT_PATH}")

### 7. Qualitative Visualizer (Compare Grayscale, Pix2Pix Colorization & Ground Truth)
This visualizer tests the generator on validation images.

In [ ]:
def visualize_pix2pix_samples(generator, dataloader, num_samples=3):
    generator.eval()
    batch = next(iter(dataloader))
    L = batch["L"].to(device)[:num_samples]
    real_ab = batch["ab"].to(device)[:num_samples]
    
    with torch.no_grad():
        pred_ab = generator(L)
        
    pred_rgbs = lab_to_rgb(L.cpu(), pred_ab.cpu())
    gt_rgbs = lab_to_rgb(L.cpu(), real_ab.cpu())
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    if num_samples == 1:
        axes = [axes]
        
    for i in range(num_samples):
        # 1. Grayscale L
        gray_img = (L[i, 0].cpu().numpy() + 1.0) / 2.0
        axes[i][0].imshow(gray_img, cmap="gray")
        axes[i][0].set_title("Grayscale Input (L)", fontsize=11, fontweight="bold")
        axes[i][0].axis("off")
        
        # 2. Pix2Pix Prediction
        axes[i][1].imshow(pred_rgbs[i])
        axes[i][1].set_title("Pix2Pix Prediction (G)", fontsize=11, fontweight="bold", color="green")
        axes[i][1].axis("off")
        
        # 3. Ground Truth
        axes[i][2].imshow(gt_rgbs[i])
        axes[i][2].set_title("Ground Truth", fontsize=11, fontweight="bold", color="navy")
        axes[i][2].axis("off")
        
    plt.tight_layout()
    plt.show()

# Run preview on current generator state (before/after training):
# visualize_pix2pix_samples(net_G, val_loader, num_samples=3)